# AI应用基础 · 工单M1.1《泵类设备故障诊断系统》 — 教师参考版
第4周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m1"], check=True)
print("数据已生成: data/")

## 参考实现 `m11`（含依赖导入）

In [ ]:
import csv
import json
import sys
import numpy as np
import numpy as np  # 通用科学计算


def m11():
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                                 recall_score)
    from sklearn.tree import DecisionTreeClassifier

    data = np.genfromtxt("data/pump_fault.csv", delimiter=",", skip_header=1)
    X, y = data[:, :5], data[:, 5].astype(int)

    idx = np.random.RandomState(42).permutation(len(X))
    X, y = X[idx], y[idx]
    n = int(len(X) * 0.75)
    Xtr, Xte, ytr, yte = X[:n], X[n:], y[:n], y[n:]

    models = {"逻辑回归": LogisticRegression(max_iter=1000),
              "决策树": DecisionTreeClassifier(max_depth=6, random_state=42),
              "随机森林": RandomForestClassifier(n_estimators=100,
                                                random_state=42)}
    print(f"{'模型':<6}{'准确':>7}{'精确':>7}{'召回':>7}{'F1':>7}")
    best = (None, 0)
    for name, m in models.items():
        m.fit(Xtr, ytr)
        p = m.predict(Xte)
        acc = accuracy_score(yte, p)
        row = [precision_score(yte, p, average="macro"),
               recall_score(yte, p, average="macro"),
               f1_score(yte, p, average="macro")]
        print(f"{name:<6}{acc:>7.3f}" + "".join(f"{v:>7.3f}" for v in row))
        if acc > best[1]:
            best = (name, acc)
    print("决策矩阵结论: 随机森林精度最优且可给出特征重要度, 选为交付模型;"
          "逻辑回归可作为产线侧轻量基线。最终:", best)


## 执行与验证

In [ ]:
m11()